# PCA state space, one subplot per seed — v8

Takes the PCA state-space figure (average trajectory per subtask, in PCA space) and **replicates it once per
seed**, five seeds side by side, laid out like the masked-model figures. This tests whether the
different geometric solutions seen so far (the "cross"/X against the "C") are a difference between **model
sizes** or simply **multiple solutions to the same task** that different seeds fall into.

Each seed gets its **own PCA frame**, fit on that seed's own pooled hidden activity, because a principal
component frame is defined only up to sign and rotation and is not shared across networks. The panels are
therefore comparable in shape, not in coordinates.

`HIDDEN` sets the model size (5 for the network in the earlier PCA figure) and `MASKED` switches between the
self-connections-only model and the standard GRU.

Loads `./generated_trials_v8`.

## 1. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8"); OUT_DIR = Path("./pca_seeds_v8"); OUT_DIR.mkdir(exist_ok=True)
T_ON, T_OFF = 10, 20
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

## 2. Load data

In [ ]:
def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    return {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"],
            "aud_int": d["aud_int"].astype(np.float32), "vis_int": d["vis_int"].astype(np.float32)}
train, test = load("train"), load("test")
T = test["X"].shape[2]; N = test["X"].shape[0]
SUBTASKS = sorted(set(test["types"]))
CONFLICT = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
print("Train", train["X"].shape, " Test", test["X"].shape)

## 3. Configuration

`HIDDEN = 5` reproduces the network from the earlier PCA figure. `MASKED = False` is the standard GRU;
set it True for the self-connections-only model.

In [ ]:
HIDDEN = 5
MASKED = False          # False = standard GRU, True = self-connections only (masked W_hh)
SEEDS  = [0, 1, 2, 3, 4]
N_EPOCHS = 50
LR = 1e-3; BATCH = 64
print("HIDDEN=%d  MASKED=%s  seeds=%s" % (HIDDEN, MASKED, SEEDS))

## 4. Models

`StandardGRU` uses `nn.GRU`. `MaskedGRU` multiplies the hidden-to-hidden weight by a fixed binary mask on
every forward pass, keeping neuron-to-self weights and zeroing neuron-to-other weights in all three gate
blocks, so the constraint holds in training and testing.

In [ ]:
class StandardGRU(nn.Module):
    def __init__(self, n_in=4, hidden=5, n_out=4):
        super().__init__(); self.H = hidden
        self.gru = nn.GRU(n_in, hidden, batch_first=True); self.readout = nn.Linear(hidden, n_out)
    def forward(self, x):
        x = x.transpose(1, 2); h, _ = self.gru(x); return self.readout(h)
    def hidden_states(self, X):
        with torch.no_grad(): h, _ = self.gru(torch.from_numpy(X).transpose(1, 2))
        return h.numpy()

class MaskedGRU(nn.Module):
    def __init__(self, n_in=4, hidden=5, n_out=4):
        super().__init__(); self.H = hidden; s = 1.0/math.sqrt(hidden)
        self.weight_ih = nn.Parameter(torch.empty(3*hidden, n_in).uniform_(-s, s))
        self.weight_hh = nn.Parameter(torch.empty(3*hidden, hidden).uniform_(-s, s))
        self.bias_ih   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.bias_hh   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.readout   = nn.Linear(hidden, n_out)
        self.register_buffer("mask", torch.eye(hidden).repeat(3, 1))
    def masked_hh(self): return self.weight_hh * self.mask
    def _run(self, x):
        B, Tt, _ = x.shape; H = self.H; Whh = self.masked_hh()
        Wir, Wiz, Win = self.weight_ih[:H], self.weight_ih[H:2*H], self.weight_ih[2*H:]
        Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
        bir, biz, bin_ = self.bias_ih[:H], self.bias_ih[H:2*H], self.bias_ih[2*H:]
        bhr, bhz, bhn = self.bias_hh[:H], self.bias_hh[H:2*H], self.bias_hh[2*H:]
        h = x.new_zeros(B, H); hs = []
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h; hs.append(h)
        return torch.stack(hs, 1)
    def forward(self, x):
        return self.readout(self._run(x.transpose(1, 2)))
    def hidden_states(self, X):
        with torch.no_grad(): h = self._run(torch.from_numpy(X).transpose(1, 2))
        return h.numpy()

def make_model(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    return (MaskedGRU if MASKED else StandardGRU)(4, HIDDEN, 4).to(device)

## 5. Train and evaluate

In [ ]:
def train_model(seed):
    model = make_model(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])),
                        batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR); loss_fn = nn.CrossEntropyLoss()
    for _ in range(N_EPOCHS):
        model.train()
        for Xb, yb in loader:
            lo = model(Xb); B, Tt, C = lo.shape
            loss = loss_fn(lo.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

def evaluate(model):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    accs = {s: float((pred[test["types"]==s]==test["y"][test["types"]==s]).mean()) for s in SUBTASKS}
    nc = float(np.mean([accs[s] for s in NONCONF]))
    return pred, accs, nc

import time
models = {}
t0 = time.time()
for s in SEEDS:
    models[s] = train_model(s)
    _, _, nc = evaluate(models[s])
    print("seed %d  non-conflict %.3f" % (s, nc))
print("trained %d seeds in %.1f min" % (len(SEEDS), (time.time()-t0)/60))

## 6. PCA per seed

Record the hidden activity across the test set as neurons x time x trials, reshape to observations x neurons,
fit a 2-component PCA on the pooled activity so every subtask shares one frame within a seed, then project
each trial's trajectory.

In [ ]:
class PCA2:
    def __init__(self, X, k=2):
        self.mean = X.mean(0)
        U, S, Vt = np.linalg.svd(X - self.mean, full_matrices=False)
        self.Vt = Vt[:k]; self.evr = (S**2)/np.sum(S**2)
    def transform(self, X): return (X - self.mean) @ self.Vt.T
    def inverse(self, Z):  return Z @ self.Vt + self.mean

def seed_pca(model):
    Hs = model.hidden_states(test["X"])              # (N, T, HIDDEN)
    p = PCA2(Hs.reshape(N*T, HIDDEN))
    Z = p.transform(Hs.reshape(N*T, HIDDEN)).reshape(N, T, 2)
    return p, Z

## 7. Colour scheme

Detection subtasks in orange, localisation in light blue, localisation conflicts in dark blue dashed,
matching the scheme used in the earlier figures.

In [ ]:
def subtask_style(s):
    if s == "det_multisensory":   return {"color": "#8c3b00", "ls": "--", "lw": 2.2}   # held-out detection conflict
    if s.startswith("det"):       return {"color": "#e0852e", "ls": "-",  "lw": 2.0}   # detection
    if "conflict" in s:           return {"color": "#12355b", "ls": "--", "lw": 2.2}   # localisation conflicts
    return {"color": "#5b9bd5", "ls": "-", "lw": 2.0}                                   # localisation

## 8. The figure: PCA state space, one panel per seed

Each panel is one seed in its own PC frame: the readout decision regions as a backdrop, one averaged
trajectory per subtask, and a star at each endpoint coloured by the class the readout assigns. Compare the
**shape** across panels, not the coordinates.

The decision-region backdrop is an approximation, a 2D slice through the full readout taken at the PCA
mean, so a trajectory can appear to cross a boundary when its off-plane components differ from the mean.

In [ ]:
fig, axes = plt.subplots(1, len(SEEDS), figsize=(4.4*len(SEEDS), 4.8))
axes = np.atleast_1d(axes)
for ax, s in zip(axes, SEEDS):
    model = models[s]
    p, Z = seed_pca(model)
    Wro = model.readout.weight.detach().numpy(); bro = model.readout.bias.detach().numpy()
    # decision-region backdrop in this seed's PC frame
    pad = 0.15
    x0, x1 = Z[:,:,0].min()-pad, Z[:,:,0].max()+pad
    y0, y1 = Z[:,:,1].min()-pad, Z[:,:,1].max()+pad
    gx = np.linspace(x0, x1, 220); gy = np.linspace(y0, y1, 220)
    GX, GY = np.meshgrid(gx, gy); flat = np.stack([GX.ravel(), GY.ravel()], 1)
    reg = np.argmax(p.inverse(flat) @ Wro.T + bro, 1).reshape(GX.shape)
    ax.pcolormesh(GX, GY, reg, cmap=ListedColormap(CLASS_COLORS), alpha=0.18, shading="auto", vmin=0, vmax=3)
    # averaged trajectory per subtask
    for sub in SUBTASKS:
        idx = np.where(test["types"] == sub)[0]
        if len(idx) == 0: continue
        tr = Z[idx].mean(0)
        st = subtask_style(sub)
        ax.plot(tr[:,0], tr[:,1], color=st["color"], ls=st["ls"], lw=st["lw"], alpha=0.9,
                label=sub if s == SEEDS[0] else None)
        cls = int(np.argmax(Wro @ p.inverse(tr[-1][None,:])[0] + bro))
        ax.scatter(*tr[-1], color=CLASS_COLORS[cls], s=70, marker="*", edgecolor="k", lw=0.5, zorder=6)
    ax.scatter(*Z[:,0,:].mean(0), color="k", s=22, zorder=7)     # common start
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_xlabel("PC 1", fontsize=9); ax.set_ylabel("PC 2", fontsize=9)
    ax.set_title("seed %d   (PC1+PC2 = %.0f%% var)" % (s, 100*p.evr[:2].sum()), fontsize=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.10))
fig.suptitle("PCA state space, average trajectory per subtask, %d hidden units%s, one panel per seed"
             % (HIDDEN, " (masked)" if MASKED else ""), fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(OUT_DIR / ("pca_%dunit%s_5seeds.png" % (HIDDEN, "_masked" if MASKED else "")),
            dpi=150, bbox_inches="tight")
plt.show()

## 9. Quantifying the solution

Classifying seeds by eye ("a cross", "a C") is replaced here by three measurements, computed in the **full
hidden space** rather than the projection, so they do not depend on the PCA frame and can be compared across
seeds:

- **axis angle**: the angle between the detection coding axis (mean state on detection trials minus mean
  state on no-detection trials) and the localisation coding axis (right minus left). 90 degrees means the
  two tasks are encoded orthogonally, which is the "cross"; a smaller angle means they interfere.
- **effective dimensionality**: the participation ratio of the endpoint cloud. Near 2 means the four classes
  span a plane (a cross); near 1 means they lie along a single curved path (a C).
- **held-out generalisation**: accuracy on `det_multisensory`, which is never trained, so it tests whether
  the code composes.

In [ ]:
def coding_axes(model):
    Hs = model.hidden_states(test["X"])[:, -1, :]        # final-timestep hidden state, (N, HIDDEN)
    ty = test["types"]
    det   = Hs[np.isin(ty, ["det_auditory_only"])].mean(0)
    nodet = Hs[np.isin(ty, ["det_absent", "det_visual_only"])].mean(0)
    right = Hs[np.isin(ty, ["loc_auditory_only_R","loc_visual_only_R","loc_multisensory_same_R"])].mean(0)
    left  = Hs[np.isin(ty, ["loc_auditory_only_L","loc_visual_only_L","loc_multisensory_same_L"])].mean(0)
    a_det = det - nodet; a_loc = right - left
    cos = float(a_det @ a_loc / (np.linalg.norm(a_det)*np.linalg.norm(a_loc) + 1e-12))
    return float(np.degrees(np.arccos(np.clip(abs(cos), 0, 1)))), a_det, a_loc

def participation_ratio(model):
    Hs = model.hidden_states(test["X"])[:, -1, :]
    C = np.cov((Hs - Hs.mean(0)).T)
    ev = np.linalg.eigvalsh(C); ev = np.clip(ev, 0, None)
    return float((ev.sum()**2) / (np.sum(ev**2) + 1e-12))

print(f"{'seed':>5} {'axis angle':>11} {'eff. dim':>9} {'non-conf':>9} {'det_multi':>10}")
print("-"*50)
rows = []
for s in SEEDS:
    ang, _, _ = coding_axes(models[s]); pr = participation_ratio(models[s])
    _, accs, nc = evaluate(models[s])
    rows.append((s, ang, pr, nc, accs["det_multisensory"]))
    print(f"{s:>5} {ang:>10.1f}° {pr:>9.2f} {nc:>9.3f} {accs['det_multisensory']:>10.3f}")
rows = np.array(rows)
print("\nmean axis angle %.1f° (90 = orthogonal coding), mean eff. dim %.2f" % (rows[:,1].mean(), rows[:,2].mean()))
if len(SEEDS) > 2:
    c = np.corrcoef(rows[:,1], rows[:,4])[0,1]
    print("corr(axis angle, det_multisensory accuracy) = %.2f   [more orthogonal -> better held-out generalisation?]" % c)

## 10. Optional: three hidden units in 3D

With three units the hidden state can be drawn directly in 3D with no projection at all, which is a
cleaner picture than PCA because nothing is discarded. Uses plotly for a rotatable plot when available and
falls back to a static matplotlib 3D view otherwise.

In [ ]:
HIDDEN_3D = 3
_h_save, _s_save = HIDDEN, SEEDS
HIDDEN, SEEDS = HIDDEN_3D, [0]
m3 = train_model(0)
Hs3 = m3.hidden_states(test["X"])                    # (N, T, 3)
HIDDEN, SEEDS = _h_save, _s_save

traj3 = {sub: Hs3[test["types"] == sub].mean(0) for sub in SUBTASKS}
try:
    import plotly.graph_objects as go
    fig3 = go.Figure()
    for sub, tr in traj3.items():
        st = subtask_style(sub)
        fig3.add_trace(go.Scatter3d(x=tr[:,0], y=tr[:,1], z=tr[:,2], mode="lines",
            line=dict(width=5, color=st["color"], dash="dash" if st["ls"]=="--" else "solid"), name=sub))
        fig3.add_trace(go.Scatter3d(x=[tr[-1,0]], y=[tr[-1,1]], z=[tr[-1,2]], mode="markers",
            marker=dict(size=5, color=st["color"]), showlegend=False))
    fig3.update_layout(title="3-unit GRU: average hidden trajectory per subtask (drag to rotate)",
        scene=dict(xaxis_title="unit 1", yaxis_title="unit 2", zaxis_title="unit 3"),
        width=900, height=700)
    fig3.show()
except ImportError:
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    fig = plt.figure(figsize=(9, 8)); ax = fig.add_subplot(111, projection="3d")
    for sub, tr in traj3.items():
        st = subtask_style(sub)
        ax.plot(tr[:,0], tr[:,1], tr[:,2], color=st["color"], ls=st["ls"], lw=st["lw"], label=sub)
        ax.scatter(*tr[-1], color=st["color"], s=40, marker="*", edgecolor="k", lw=0.4)
    ax.set_xlabel("unit 1"); ax.set_ylabel("unit 2"); ax.set_zlabel("unit 3")
    ax.set_title("3-unit GRU: average hidden trajectory per subtask")
    ax.legend(loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=7, frameon=False)
    plt.tight_layout(); plt.savefig(OUT_DIR / "trajectories_3unit_3d.png", dpi=150, bbox_inches="tight"); plt.show()
    print("plotly not installed, drew a static 3D view instead. For a rotatable plot: pip install plotly")

## Notes

- Each panel uses its own PCA frame, so compare shapes and not coordinates. A PC frame is defined only up
  to sign and rotation, which is why the seeds cannot be overlaid on shared axes.
- The question this figure answers: are the different geometries a property of model size, or are they
  multiple solutions the task admits, with the seed (weight initialisation and trial order) deciding which
  one is found?
- The numbers in section 9 are computed in the full hidden space, so they are independent of the
  projection and can be compared directly across seeds and against the masked models.